# Objective 1：2026 年 r/jobs 评论采集与清理

本阶段只完成数据取得、清理、每月抽样、质量检查与交接。TF-IDF、PCA、K-means 等后续单元格保留为可选内容，默认不运行。

目标为 2026 年 1 月至采集截止时刻，每月 1,000 条清理后合格评论。2026 年 10 月是部分月份；11—12 月尚未发生。

数据源为 Arctic Shift 公开历史评论接口。每月划分 20 个等长 UTC 时间段，在各段用固定种子选择随机起点读取候选，清理后各段抽取 50 条。
这能分散月份内的时间覆盖，但不是全月评论的简单随机样本；每条评论的入样概率仍受局部活跃度影响。无关键词筛选，也不按毒性类别筛选。

实际数量、时间范围、清理排除数和文件校验值以运行目录的 CSV、manifest 与采集记录为准。等额月样本不能估计全站绝对讨论量。

已完成：2026 年 1—10 月各 1,000 条，合计 10,000 条。采集截止 2026-10-28T07:10:54+00:00（UTC），10 月为部分月份。原始候选 20,213 条，清理排除 2,447 条；数据单元格已顺序执行，未运行主题模型。

## 1. 环境与参数

从项目根目录或 `notebook/` 启动均可。复用 `functions.preprocessing.cleanup_text` 和
`functions.reddit_collection.read_comments`；新增的 `topic_data` 模块保留独立的主题语料规则。
不导入 TensorFlow，不下载 GloVe，不修改毒性模型、tokenizer 或阈值。

默认仅执行数据检查。网络采集和主题模型均为显式开关，避免打开 notebook 时意外启动长任务。
编辑共享模块后需要重启 kernel。以下代码尚未在真实的每月 1,000 条语料上运行。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'functions').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'functions' / 'topic_data.py').exists():
    raise RuntimeError('Start this notebook from the project root or notebook directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from functions.reddit_collection import read_comments
from functions.preprocessing import cleanup_text
from functions.topic_data import month_windows, export_topic_run

SEED = 42
YEAR = 2026
PER_MONTH = 1000
MIN_WORDS = 3
AS_OF = pd.Timestamp.now(tz='UTC').floor('s').isoformat()
DATA_ROOT = ROOT / 'dataset' / 'reddit' / 'topics_2026'

# 可指定已有运行目录；None 会查找最新运行记录。
RUN_DIR = ROOT / 'dataset/reddit/topics_2026/time_stratified_20261028/prepared'
RAW_EXPORT_PATH = None  # 本地历史导出 CSV 的 Path；至少含四个必需字段
RUN_ARCHIVE_DOWNLOAD = False
RUN_TOPIC_MODEL = False
SELECTED_K = None  # 阅读选 k 图和评论后填写，例如整数 6；不自动决定
TOPIC_LABELS = {}  # 人工审阅后填写 {cluster_id: '主题名称'}
SAVE_TOPIC_OUTPUTS = False
LAYOFF_CSV = None  # 可选 CSV，列为 month,laid_off
LAYOFF_CLUSTER_IDS = []  # 根据主题内容确定，不根据外部曲线反向挑选

versions = {name: importlib.metadata.version(name)
            for name in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']}
print('Python:', sys.version.split()[0])
print(versions)
print('UTC cutoff for a new acquisition:', AS_OF)
display(pd.DataFrame(month_windows(YEAR, AS_OF)))

## 2. 数据来源、时间分层与采集

复用共享的 CSV 读取与文本清理代码；历史评论由 `scripts/collect_monthly_topics.py` 获取。
既有 `collect_reddit.py collect` 仅用于近期列表，不用于历史月份回溯。

每月分成 20 个等长 UTC 时间段，以 seed=42 为基础在每段选随机起点，读取有限候选批次；必要时沿时间分页或回到该段起点补充。
先过滤与清理，再在每段合格候选中无放回抽取 50 条。请求逐条限速，按时间段保存检查点，失败可从同一工作目录恢复。

这是随机起点的时间分层候选抽样，不声称读全该月，也不声称全月简单随机或全站代表性。
`source_window_complete=False` 是本设计的明确记录，不表示请求失败。来源偏差、未知机器人和未验证语言仍需报告。

如重新采集，将 `RUN_ARCHIVE_DOWNLOAD=True`；会使用新的工作目录。若读本地历史导出，填写 `RAW_EXPORT_PATH`。
必需列：`comment_id,text,created_utc,subreddit`；发表时间为 UTC Unix 秒。

In [ ]:
if RAW_EXPORT_PATH is not None and RUN_ARCHIVE_DOWNLOAD:
    raise ValueError('Choose a local export or an archive download for this run.')

if RAW_EXPORT_PATH is not None or RUN_ARCHIVE_DOWNLOAD:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    RUN_DIR = DATA_ROOT / ('import_' + stamp if RAW_EXPORT_PATH is not None else 'archive_' + stamp)
    if RAW_EXPORT_PATH is not None:
        raw_path = Path(RAW_EXPORT_PATH)
        records = read_comments(raw_path)
        export_topic_run(
            RUN_DIR, records, YEAR, AS_OF,
            {'method': 'local_csv', 'file': raw_path.name,
             'sha256': hashlib.sha256(raw_path.read_bytes()).hexdigest(),
             'coverage': 'not independently verified'},
            per_month=PER_MONTH, seed=SEED, min_words=MIN_WORDS,
        )
    else:
        work_dir = RUN_DIR
        result = subprocess.run(
            [sys.executable, str(ROOT / 'scripts' / 'collect_monthly_topics.py'),
             '--work-dir', str(work_dir), '--year', str(YEAR),
             '--per-month', str(PER_MONTH), '--seed', str(SEED),
             '--min-words', str(MIN_WORDS), '--as-of', AS_OF],
            cwd=ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace',
        )
        RUN_DIR = work_dir / 'prepared'
        print(result.stdout)
        if result.returncode:
            print('Acquisition did not finish. Inspect its status report:', result.stderr)

if RUN_DIR is None:
    runs = [path for path in DATA_ROOT.glob('*') if path.is_dir()]
    RUN_DIR = max(runs, key=lambda path: path.stat().st_mtime) if runs else None
elif not Path(RUN_DIR).is_absolute():
    RUN_DIR = ROOT / RUN_DIR
print('Selected run:', RUN_DIR)

## 3. 检查实际采集状态

`source_error` 或 `not_attempted` 表示缺数据，不是零评论。
`source_window_complete` 仅指相应来源的窗口是否读取完整；公开档案也可能有收录缺失。
时间范围应以该次运行的 manifest 为准，而不是今天的日期。

In [ ]:
corpus = pd.DataFrame()
coverage = pd.DataFrame()
manifest = {}
acquisition = {}
if RUN_DIR is not None:
    RUN_DIR = Path(RUN_DIR)
    status_path = RUN_DIR / 'acquisition_status.json'
    if not status_path.exists():
        status_path = RUN_DIR.parent / 'acquisition_status.json'
    if status_path.exists():
        acquisition = json.loads(status_path.read_text(encoding='utf-8'))
        if 'months' in acquisition:
            display(pd.DataFrame(acquisition['months']))
        elif 'strata' in acquisition:
            progress = pd.DataFrame(acquisition['strata'])
            if not progress.empty:
                display(progress.groupby('month').agg(
                    completed_strata=('complete', 'sum'),
                    eligible_candidates=('eligible_candidates', 'sum'),
                    target_samples=('quota', 'sum'),
                ))
        print('Retrieval status:', acquisition['status'])
    manifest_path = RUN_DIR / 'sampling_manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        for name, expected_hash in manifest['sha256'].items():
            actual_hash = hashlib.sha256((RUN_DIR / name).read_bytes()).hexdigest()
            if actual_hash != expected_hash:
                raise ValueError(f'File changed since preparation: {name}')
        if (manifest['year'], manifest['target_per_month'], manifest['seed'], manifest['min_words']) != (
            YEAR, PER_MONTH, SEED, MIN_WORDS
        ):
            raise ValueError('Selected run uses different parameters. Select or prepare the intended run.')
        print('Run cutoff:', manifest['as_of_utc'])
        corpus = pd.read_csv(RUN_DIR / 'comments_clean.csv', dtype={'comment_id': str}, keep_default_na=False)
        coverage = pd.read_csv(RUN_DIR / 'monthly_coverage.csv', keep_default_na=False)
        display(coverage)
        display(pd.DataFrame([manifest['counts']]))
        if not corpus.empty:
            assert corpus['comment_id'].is_unique
            assert corpus['text'].str.strip().ne('').all()
            corpus['created_at'] = pd.to_datetime(corpus['created_at'], utc=True)
            assert corpus.groupby('month').size().le(PER_MONTH).all()
            assert len(corpus) == manifest['counts']['selected']
            if manifest.get('strata_per_month') == 20:
                strata_counts = corpus.groupby(['month', 'sampling_stratum']).size()
                assert strata_counts.eq(PER_MONTH // 20).all()
                assert corpus.groupby('month')['sampling_stratum'].nunique().eq(20).all()
                assert coverage['selected'].eq(PER_MONTH).all()

if corpus.empty:
    print('No prepared historical corpus is available. Modelling cells will skip; no substitute data is used.')

## 4. 清理规则与审计

1. 检查 ID、r/jobs 来源与 UTC 秒时间；排除范围外或未来评论。
2. 排除缺失正文、`[deleted]`、`[removed]`、已知机器人和来源明确标注的非英语内容。
3. 按 ID 去重；同 ID 的正文或发表时间冲突则停止，要求核对来源。
4. 保留 `text` 原文；HTML 实体解码、小写化后调用共享 `cleanup_text`，生成 `text_topic`。
5. 默认至少保留 3 个英文词；该阈值可调整，但需重新生成抽样与模型。
6. 不同 ID 的相同文本保留，并标记 `duplicate_text`；不按正文全局去重。
7. 在每个时间段合格候选中无放回抽取 50 条，20 段合计每月 1,000 条，seed=42；不足时报告缺口，不重复补齐。

语言缺失的评论仍需要人工抽查。不要把脏话、负面情绪或裁员词当作删除条件。
`annotation_notes` 和 `is_toxic` 不进入主题特征。

In [ ]:
if manifest:
    audit = pd.read_csv(RUN_DIR / 'cleaning_audit.csv')
    excluded = pd.read_csv(RUN_DIR / 'excluded_rows.csv', keep_default_na=False)
    display(audit)
    if not excluded.empty:
        display(excluded.groupby(['month', 'exclusion_reason'], dropna=False).size().rename('n_rows').reset_index())
    assert int(audit['n_rows'].sum()) == manifest['counts']['raw']
    assert manifest['counts']['eligible'] + manifest['counts']['excluded'] == manifest['counts']['raw']
if not corpus.empty:
    display(corpus[['comment_id', 'month', 'text', 'text_topic', 'word_count',
                    'duplicate_text', 'language_status']].sample(min(10, len(corpus)), random_state=SEED))
    print('Review these random examples for language, bot/template text and over-cleaning.')

## 5. 样本覆盖与文本长度

这些图显示本次样本的覆盖和长度。每月固定 1,000 条会人为固定月总量，因此后续主要比较主题占比。
没有采集到的数据不补零；10 月需标注为部分月份。
同一帖子贡献过多时，在报告中说明，并考虑按帖子限制的敏感性分析。

In [ ]:
if not corpus.empty:
    monthly_summary = corpus.groupby('month').agg(n_comments=('comment_id', 'size'),
                                                  n_posts=('link_id', 'nunique'))
    display(monthly_summary)
    display(corpus['link_id'].replace('', np.nan).value_counts().head(10).rename('n_comments').to_frame())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    monthly_summary['n_comments'].plot.bar(ax=axes[0], title='Sample coverage per UTC month')
    axes[0].set_ylabel('Sampled comments (not subreddit volume)')
    corpus['word_count'].plot.hist(bins=30, ax=axes[1], title='Cleaned word counts')
    fig.tight_layout()
    plt.show()

## 6. TF-IDF → PCA（50 维）

以下是可选建模单元格。取得语料并检查后将 `RUN_TOPIC_MODEL=True`，再从设置单元格顺序执行。
TF-IDF 使用 unigram/bigram、`min_df=3`、`max_df=0.9`、最多 10,000 特征。
保留 `no/not/nor/off`，避免丢失否定信息和 `laid off` 中的 off。

全零 TF-IDF 评论不参与聚类，导出时保留 ID 和排除原因。
PCA 使用支持稀疏输入的 ARPACK solver，不将整个词矩阵转为 dense。
50 必须严格小于样本数和特征数；不把 TruncatedSVD 无说明地替代 PCA。
此处一次性在整个回顾性语料上拟合，所有月份共享同一套主题模型。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from threadpoolctl import threadpool_limits

features_ready = False
final_ready = False
if RUN_TOPIC_MODEL and not corpus.empty:
    stop_words = sorted(set(ENGLISH_STOP_WORDS) - {'no', 'not', 'nor', 'off'})
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2), min_df=3, max_df=0.90, max_features=10000,
        stop_words=stop_words, sublinear_tf=True, dtype=np.float32,
    )
    X_all = vectorizer.fit_transform(corpus['text_topic'])
    has_features = np.asarray(X_all.getnnz(axis=1)).ravel() > 0
    feature_excluded = corpus.loc[~has_features, ['comment_id', 'month']].assign(reason='zero_tfidf')
    model_df = corpus.loc[has_features].reset_index(drop=True)
    X = X_all[has_features]
    if min(X.shape) <= 50:
        raise ValueError(f'PCA(50) needs >50 samples and features; actual shape={X.shape}.')
    pca = PCA(n_components=50, svd_solver='arpack', random_state=SEED)
    with threadpool_limits(limits=2):
        Z = pca.fit_transform(X)
    print('TF-IDF:', X.shape, 'PCA:', Z.shape)
    print('PCA explained variance:', pca.explained_variance_ratio_.sum())
    print('Excluded zero-vector comments:', len(feature_excluded))
    plt.plot(np.arange(1, 51), np.cumsum(pca.explained_variance_ratio_))
    plt.xlabel('Principal components')
    plt.ylabel('Cumulative explained variance')
    plt.show()
    features_ready = True
else:
    print('Topic modelling is disabled or no corpus is available.')

## 7. 选择簇数 k

在同一个 50 维空间中比较 k=2…10，显式使用 `n_init=10`。
Silhouette 使用固定的最多 2,000 条样本；不能跨不同特征空间直接比较其数值。
同时检查 Elbow、Silhouette、簇大小和可解释性。不要为了得到“裁员簇”反复调 k。
若计算超过几分钟，请先按仓库约定确认长任务安排。

In [ ]:
if features_ready:
    eval_indices = np.random.default_rng(SEED).choice(len(Z), min(2000, len(Z)), replace=False)
    k_models = {}
    k_rows = []
    with threadpool_limits(limits=2):
        for k in range(2, min(10, len(Z) - 1) + 1):
            fitted = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z)
            labels_eval = fitted.labels_[eval_indices]
            unique_labels = np.unique(labels_eval)
            score = (silhouette_score(Z[eval_indices], labels_eval)
                     if 1 < len(unique_labels) < len(eval_indices) else np.nan)
            sizes = pd.Series(fitted.labels_).value_counts()
            k_models[k] = fitted
            k_rows.append({'k': k, 'inertia': fitted.inertia_, 'silhouette': score,
                           'actual_clusters': len(sizes), 'smallest_cluster': sizes.min(),
                           'largest_cluster_share': sizes.max() / len(Z)})
    k_results = pd.DataFrame(k_rows)
    display(k_results)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    k_results.plot(x='k', y='inertia', marker='o', ax=axes[0], title='Elbow')
    k_results.plot(x='k', y='silhouette', marker='o', ax=axes[1], title='Silhouette: fixed evaluation subset')
    fig.tight_layout()
    plt.show()
    print('Inspect these results and comments, then set SELECTED_K in the setup cell.')

## 8. 原词空间主题画像与人工命名

`SELECTED_K` 确认后，计算每簇原始 TF-IDF 向量均值，取前 15 个词。
查看最接近中心的 5 条评论及随机 5 条评论再命名；不把 50 维 PCA 坐标直接解释成词。
`TOPIC_LABELS` 留空时名称明确标记为未审阅。

In [ ]:
if features_ready and SELECTED_K is not None:
    if SELECTED_K not in k_models:
        raise ValueError('SELECTED_K must be one of the evaluated values.')
    final_model = k_models[SELECTED_K]
    model_df['cluster'] = final_model.labels_
    terms = vectorizer.get_feature_names_out()
    profiles = []
    representatives = []
    for cluster_id in sorted(model_df['cluster'].unique()):
        mask = model_df['cluster'].eq(cluster_id).to_numpy()
        indices = np.flatnonzero(mask)
        weights = np.asarray(X[mask].mean(axis=0)).ravel()
        top_terms = terms[weights.argsort()[::-1][:15]].tolist()
        label = TOPIC_LABELS.get(int(cluster_id), f'Unreviewed cluster {cluster_id}')
        profiles.append({'cluster': cluster_id, 'topic_label': label,
                         'n_comments': int(mask.sum()), 'top_terms': ', '.join(top_terms),
                         'label_reviewed': int(cluster_id) in TOPIC_LABELS})
        distances = np.linalg.norm(Z[indices] - final_model.cluster_centers_[cluster_id], axis=1)
        nearest = indices[np.argsort(distances)[:5]]
        random_rows = model_df.loc[mask].sample(min(5, len(indices)), random_state=SEED).index.to_numpy()
        for kind, selected in [('nearest', nearest), ('random', random_rows)]:
            part = model_df.loc[selected, ['comment_id', 'text', 'month', 'cluster']].copy()
            part['selection'] = kind
            representatives.append(part)
    topic_profiles = pd.DataFrame(profiles)
    representative_comments = pd.concat(representatives, ignore_index=True)
    model_df['topic_label'] = model_df['cluster'].map(topic_profiles.set_index('cluster')['topic_label'])
    display(topic_profiles)
    display(representative_comments)
    final_ready = True
elif features_ready:
    print('Final clustering awaits a reviewed SELECTED_K; no default k was silently chosen.')

## 9. 月度主题占比

`n_topic` 为抽样后可分配主题的评论数，`n_assigned` 为该月所有可分配主题的评论数；
主题占比 = `n_topic / n_assigned`。全零特征评论另行报告，月份缺失不补为全站零活动。
10 月或其他当前月份是部分月份。每月等额抽样不能反映主题的全站绝对数量变化。

In [ ]:
if final_ready:
    topic_counts = pd.crosstab(model_df['month'], model_df['cluster']).sort_index()
    topic_shares = topic_counts.div(topic_counts.sum(axis=1), axis=0)
    monthly_topics = topic_counts.rename_axis(index='month', columns='cluster').stack().rename('n_topic').reset_index()
    monthly_topics['n_assigned'] = monthly_topics['month'].map(topic_counts.sum(axis=1))
    monthly_topics['topic_share'] = monthly_topics['n_topic'] / monthly_topics['n_assigned']
    monthly_topics['topic_label'] = monthly_topics['cluster'].map(topic_profiles.set_index('cluster')['topic_label'])
    monthly_topics = monthly_topics.merge(coverage[['month', 'partial_month', 'source_window_complete']],
                                         on='month', how='left', validate='many_to_one')
    display(monthly_topics)
    assert np.allclose(topic_shares.sum(axis=1), 1)
    topic_shares.plot(figsize=(12, 5), marker='o', title='Sample topic shares (current month may be partial)')
    plt.ylabel('Share among assigned comments')
    plt.show()
    fig, ax = plt.subplots(figsize=(12, 5))
    chart = ax.imshow(topic_shares.T.to_numpy(), aspect='auto', vmin=0, vmax=1, cmap='Blues')
    ax.set_xticks(range(len(topic_shares)), topic_shares.index, rotation=45)
    ax.set_yticks(range(len(topic_shares.columns)), topic_shares.columns)
    ax.set(xlabel='UTC month', ylabel='Cluster', title='Sample topic shares')
    fig.colorbar(chart, ax=ax, label='Share')
    fig.tight_layout()
    plt.show()

## 10. 与 layoffs.fyi 的探索性对比

在命名完成后，根据内容填写 `LAYOFF_CLUSTER_IDS`，不能根据外部曲线挑簇。
外部 CSV 约定 `month` 为 YYYY-MM、`laid_off` 为该月裁员人数；单位与来源需另行记录。
只比较重叠的完整月份，排除本次采集的部分月份。标准化仅帮助看走势，不表示单位相同。
layoffs.fyi 主要覆盖科技行业，r/jobs 主题范围更广；短序列同期相关不能证明因果或预警能力。

In [ ]:
if final_ready and LAYOFF_CSV is not None and LAYOFF_CLUSTER_IDS:
    invalid_clusters = set(LAYOFF_CLUSTER_IDS) - set(topic_shares.columns)
    if invalid_clusters:
        raise ValueError(f'Unknown layoff clusters: {invalid_clusters}')
    external = pd.read_csv(LAYOFF_CSV, dtype={'month': str})
    if not {'month', 'laid_off'}.issubset(external.columns) or external['month'].duplicated().any():
        raise ValueError('Expected one external row per month with month,laid_off columns.')
    external['laid_off'] = pd.to_numeric(external['laid_off'], errors='raise')
    series = topic_shares[LAYOFF_CLUSTER_IDS].sum(axis=1).rename('layoff_topic_share').reset_index()
    comparison = series.merge(external[['month', 'laid_off']], on='month', how='inner', validate='one_to_one')
    partial = coverage['partial_month'].astype(str).str.lower().eq('true')
    comparison = comparison.loc[~comparison['month'].isin(coverage.loc[partial, 'month'])].set_index('month')
    display(comparison)
    if len(comparison) >= 3 and comparison.std().gt(0).all():
        ((comparison - comparison.mean()) / comparison.std()).plot(figsize=(11, 4), marker='o', title='Exploratory comparison: standardized monthly series')
        plt.ylabel('Standardized value')
        plt.show()
    else:
        print('Too few overlapping complete months or a constant series; no comparison plot.')

## 11. 导出与交接

检查 k、主题名称与数据覆盖后可将 `SAVE_TOPIC_OUTPUTS=True`。
新建时间戳目录；输出只留在 Git 忽略的数据目录，不写入 `models/`。
主题文件以 `comment_id` 与原始评论、毒性预测关联；全零特征行保留，并标记未分配原因。
保存参数、版本、来源 manifest 和人工命名状态，避免 notebook 输出被 nbstripout 清除后失去证据。

In [ ]:
if final_ready and SAVE_TOPIC_OUTPUTS:
    output = RUN_DIR / ('topics_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir()
    assignments = model_df[['comment_id', 'cluster', 'topic_label']]
    comment_topics = corpus[['comment_id', 'month']].merge(assignments, on='comment_id', how='left', validate='one_to_one')
    comment_topics['topic_status'] = np.where(comment_topics['cluster'].notna(), 'assigned', 'zero_tfidf')
    comment_topics.to_csv(output / 'comment_topics.csv', index=False, encoding='utf-8-sig')
    topic_profiles.to_csv(output / 'topic_profiles.csv', index=False, encoding='utf-8-sig')
    monthly_topics.to_csv(output / 'monthly_topic_trends.csv', index=False, encoding='utf-8-sig')
    feature_excluded.to_csv(output / 'zero_tfidf_comments.csv', index=False, encoding='utf-8-sig')
    representative_comments.to_csv(output / 'representative_comments.csv', index=False, encoding='utf-8-sig')
    k_results.to_csv(output / 'k_selection.csv', index=False)
    parameters = {
        'seed': SEED, 'selected_k': SELECTED_K, 'pca_components': 50,
        'explained_variance': float(pca.explained_variance_ratio_.sum()),
        'versions': versions, 'topic_labels': TOPIC_LABELS,
        'all_labels_reviewed': bool(topic_profiles['label_reviewed'].all()),
        'tfidf': {'ngram_range': [1, 2], 'min_df': 3, 'max_df': 0.90,
                  'max_features': 10000, 'sublinear_tf': True,
                  'stop_words': stop_words, 'dtype': 'float32'},
        'source_manifest': manifest,
    }
    (output / 'run_parameters.json').write_text(json.dumps(parameters, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved topic artifacts:', output)

## 本阶段交接与局限

本阶段交付：原始候选、清理后每月样本、排除行及原因、清理计数、月度覆盖、抽样 manifest、时间段请求与检查点。
在下一阶段开始前，人工抽查语言、机器人/模板内容、同文不同 ID 与单一帖子的占比。

- 每月 1,000 条是本次设计的配额，不是该月全站评论总量。
- 时间分层随机起点候选抽样并非全月简单随机；档案收录可能不完整。
- 来源缺少语言标签时标为 unverified，不将其误报为已通过自动英语检测。
- 当前月份为部分月份，不与完整月份的绝对总量直接比较。
- 主题模型保持关闭；当前没有训练聚类、选择 k 或编造主题结论。

参考：[Arctic Shift API](https://github.com/ArthurHeitmann/arctic_shift/blob/master/api/README.md)、项目 README 和 proposal。
